# Feature Engineering — Online Retail II
## Assignment 2 : Nettoyage, transformations et features RFM

Ce notebook exécute et visualise toutes les étapes de transformation des données.

## 0. Imports

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from src.data import load_data, clean_data, compute_rfm, remove_outliers, normalize_rfm, full_pipeline

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print('Imports OK')

## 1. Chargement et nettoyage

In [ ]:
DATA_PATH = '../data/online_retail_II.csv'

df_raw   = load_data(DATA_PATH)
df_clean = clean_data(df_raw)

print(f'\nRéduction : {len(df_raw):,} → {len(df_clean):,} lignes ({(1 - len(df_clean)/len(df_raw))*100:.1f}% supprimées)')

## 2. Calcul des features RFM brutes

In [ ]:
rfm = compute_rfm(df_clean)
rfm.describe().round(2)

## 3. Visualisation des distributions brutes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['steelblue', 'coral', 'mediumseagreen']

for i, (col, color) in enumerate(zip(['Recency', 'Frequency', 'Monetary'], colors)):
    axes[i].hist(rfm[col], bins=60, color=color, edgecolor='white')
    axes[i].set_title(f'Distribution brute — {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Nb clients')

plt.suptitle('Distributions RFM AVANT transformations', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Traitement des outliers

In [ ]:
rfm_clean = remove_outliers(rfm, upper_quantile=0.99)

# Comparaison avant/après
print('\n--- Avant suppression outliers ---')
print(rfm[['Recency','Frequency','Monetary']].describe().round(2))
print('\n--- Après suppression outliers ---')
print(rfm_clean[['Recency','Frequency','Monetary']].describe().round(2))

## 5. Log-transformation

In [ ]:
rfm_log = rfm_clean[['Recency', 'Frequency', 'Monetary']].copy()
rfm_log['Recency']   = np.log1p(rfm_log['Recency'])
rfm_log['Frequency'] = np.log1p(rfm_log['Frequency'])
rfm_log['Monetary']  = np.log1p(rfm_log['Monetary'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['steelblue', 'coral', 'mediumseagreen']

for i, (col, color) in enumerate(zip(['Recency', 'Frequency', 'Monetary'], colors)):
    axes[i].hist(rfm_log[col], bins=60, color=color, edgecolor='white')
    axes[i].set_title(f'Après log1p — {col}', fontweight='bold')
    axes[i].set_xlabel(f'log1p({col})')
    axes[i].set_ylabel('Nb clients')

plt.suptitle('Distributions RFM APRÈS log-transformation', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Normalisation complète (StandardScaler)

In [ ]:
rfm_scaled, scaler = normalize_rfm(rfm_clean)

print('\nStatistiques du dataset normalisé (doit être ~0 mean, ~1 std) :')
print(rfm_scaled[['R_scaled', 'F_scaled', 'M_scaled']].describe().round(3))

## 7. Visualisation finale : avant vs après

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
colors = ['steelblue', 'coral', 'mediumseagreen']
raw_cols    = ['Recency', 'Frequency', 'Monetary']
scaled_cols = ['R_scaled', 'F_scaled', 'M_scaled']

for i, (raw, scaled, color) in enumerate(zip(raw_cols, scaled_cols, colors)):
    axes[0, i].hist(rfm_clean[raw], bins=60, color=color, edgecolor='white', alpha=0.8)
    axes[0, i].set_title(f'AVANT — {raw}', fontweight='bold')

    axes[1, i].hist(rfm_scaled[scaled], bins=60, color=color, edgecolor='white', alpha=0.8)
    axes[1, i].set_title(f'APRÈS normalisation — {scaled}', fontweight='bold')

plt.suptitle('Comparaison distributions RFM : avant vs après transformations',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Scatter plots RFM normalisé

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

pairs = [('R_scaled', 'F_scaled'), ('R_scaled', 'M_scaled'), ('F_scaled', 'M_scaled')]

for ax, (x, y) in zip(axes, pairs):
    ax.scatter(rfm_scaled[x], rfm_scaled[y], alpha=0.3, s=5, color='steelblue')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f'{x} vs {y}', fontweight='bold')

plt.suptitle('Scatter plots — Features RFM normalisées', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Synthèse

| Étape | Lignes/Clients | Remarque |
|-------|---------------|----------|
| Dataset brut | ~1 067 371 lignes | |
| Après nettoyage | ~750 000 lignes | Suppression guests, annulations, invalides |
| RFM brut | ~5 800 clients | Agrégation par Customer ID |
| Après outliers | ~5 742 clients | Suppression 99e percentile |
| RFM normalisé | ~5 742 clients | log1p + StandardScaler |

**Le dataset `rfm_scaled` est prêt pour le clustering (Assignment 3).**